In [ ]:
# Importing the required libraries
import pandas as pd
from google.colab import files


# PRIVATE

In [ ]:
priv_df = pd.read_csv("/content/2017_Private_WAEC_Data.csv", header=None)


In [ ]:
# Viewing the DataFrame
priv_df.head(10)


In [ ]:
# Renaming the columns
priv_df.columns = [
    "state",
    "cands_male",
    "cands_female",
    "cands_total",
    "creds_5_eng_male",
    "creds_5_eng_female",
    "creds_5_eng_total",
    "creds_5_maths_male",
    "creds_5_maths_female",
    "creds_5_maths_total",
    "creds_5_eng_maths_male",
    "creds_5_eng_maths_female",
    "creds_5_eng_maths_total",
    "creds_5_eng_maths_male_pct",
    "creds_5_eng_maths_female_pct",
    "creds_5_eng_maths_total_pct"
]


## Defining a function to handle state names

This goes a little further than the 2018 notebook's `state_handling`
function. Rather than only catching the exact text "Abuja", it treats
**any** state value that contains "abuja" as a substring — in any
case, and regardless of what else is in the cell, e.g. "Abuja",
"ABUJA", "FCT-Abuja", "Abuja-FCT" — as a reference to the FCT, and
normalizes all of them to a single, cleanly-cased "FCT". It also
catches "fct" typed in the wrong case (e.g. "fct"), and strips stray
whitespace before comparing, since one of this year's source files has
state names with leading/trailing spaces.

It also drops "Offshore" (present in the public files, not the
private ones) before separating out the "Nigeria" grand-total row,
sorting the remaining states alphabetically, and appending "Nigeria"
back as the last row — the same overall approach as the 2018
notebook's `state_handling`.

In [ ]:
def state_handling(df):
  df["state"] = df["state"].astype("string").str.strip().str.title()

  # Normalizing any Abuja/FCT variant (any case, any wording that
  # references Abuja or FCT) to a single clean "FCT" label
  is_fct_like = (
      df["state"].str.contains("abuja", case=False, na=False) |
      df["state"].str.fullmatch("fct", case=False, na=False)
  )
  df.loc[is_fct_like, "state"] = "FCT"

  # Removing Offshore, if present
  df = df[
      ~df["state"].str.fullmatch("offshore", case=False, na=False)
  ]

  nigeria_row = df[df["state"] == "Nigeria"]
  df = df[df["state"] != "Nigeria"]

  df = df.sort_values("state")

  df = pd.concat([df, nigeria_row])

  df = df.reset_index(drop=True)

  return df


## Defining a function to handle the numerical columns

In [ ]:
def numeric_handling(df, num_cols):
  for col in num_cols:
    df[col] = df[col].astype("string").str.replace(",", "", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

  return df


In [ ]:
# Viewing the result
priv_df


## Dropping the irrelevant rows

The first five rows of the raw sheet are title/header clutter (the WAEC
letterhead lines, the merged category headers, and the MALE/FEMALE/TOTAL
sub-header row) rather than actual state data, so they're dropped here
before any cleaning runs.

In [ ]:
# Dropping the irrelevant rows
priv_df = priv_df.drop(axis=0, index=[0, 1, 2, 3, 4]).reset_index(drop=True)


In [ ]:
# Viewing the result
priv_df


In [ ]:
priv_df = state_handling(priv_df)


In [ ]:
# Viewing the result
priv_df


In [ ]:
# Handling the numerical columns
num_cols = [
    "cands_male",
    "cands_female",
    "cands_total",
    "creds_5_eng_male",
    "creds_5_eng_female",
    "creds_5_eng_total",
    "creds_5_maths_male",
    "creds_5_maths_female",
    "creds_5_maths_total",
    "creds_5_eng_maths_male",
    "creds_5_eng_maths_female",
    "creds_5_eng_maths_total",
    "creds_5_eng_maths_male_pct",
    "creds_5_eng_maths_female_pct",
    "creds_5_eng_maths_total_pct"
]
priv_df = numeric_handling(priv_df, num_cols)


In [ ]:
# Viewing the result
priv_df


In [ ]:
# Saving the DataFrame
file_name = "Cleaned 2017 Private WAEC Data.csv"
priv_df.to_csv(file_name, index=False)
files.download(file_name)


# PUBLIC

This file is loaded the same way as the private one above — with
`header=None` — so the raw row/column layout lines up with `priv_df`
exactly. (The 2018 base notebook's public-side cell accidentally omitted
`header=None`, which shifted every row by one and made the later
row-dropping step rely on hardcoded positions found by trial and error.
Loading it the same way as the private file avoids that entirely.)

In [ ]:
pub_df = pd.read_csv("/content/2017_Public_WAEC_Data.csv", header=None)


In [ ]:
# Viewing the DataFrame
pub_df.head(10)


In [ ]:
# Renaming the columns
pub_df.columns = [
    "state",
    "cands_male",
    "cands_female",
    "cands_total",
    "creds_5_eng_male",
    "creds_5_eng_female",
    "creds_5_eng_total",
    "creds_5_maths_male",
    "creds_5_maths_female",
    "creds_5_maths_total",
    "creds_5_eng_maths_male",
    "creds_5_eng_maths_female",
    "creds_5_eng_maths_total",
    "creds_5_eng_maths_male_pct",
    "creds_5_eng_maths_female_pct",
    "creds_5_eng_maths_total_pct"
]


In [ ]:
# Viewing the result
pub_df


In [ ]:
# Dropping the irrelevant rows
pub_df = pub_df.drop(axis=0, index=[0, 1, 2, 3, 4]).reset_index(drop=True)


In [ ]:
# Viewing the result
pub_df.head(10)


In [ ]:
pub_df = state_handling(pub_df)


In [ ]:
pub_df


In [ ]:
# Handling the numeric columns
num_cols = [
    "cands_male",
    "cands_female",
    "cands_total",
    "creds_5_eng_male",
    "creds_5_eng_female",
    "creds_5_eng_total",
    "creds_5_maths_male",
    "creds_5_maths_female",
    "creds_5_maths_total",
    "creds_5_eng_maths_male",
    "creds_5_eng_maths_female",
    "creds_5_eng_maths_total",
    "creds_5_eng_maths_male_pct",
    "creds_5_eng_maths_female_pct",
    "creds_5_eng_maths_total_pct"
]

pub_df = numeric_handling(pub_df, num_cols)


In [ ]:
# Viewing the result
pub_df


In [ ]:
# Saving the result
file_name = "Cleaned 2017 Public WAEC Data.csv"

pub_df.to_csv(file_name, index=False)
files.download(file_name)
